# Database reader: outliers, hypervolume, and the observed Pareto front

Loads the raw MOBO simulation database, removes outlier simulations, and runs the correlation/hypervolume/Pareto-front analysis. All the underlying logic lives in `iceburner.database` and `iceburner.pareto`; this notebook just calls it and displays the results.

Produces `results/pareto_dominating_points.csv`, used by `dominating_points.ipynb`.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
from iceburner import database, pareto

df = database.load_state_dataframe("../data/Gorgon_MF_DB_Gorgon_MF_iceburner_final_week.db")
df, outliers = database.filter_outliers(df)
outliers

## Correlation and hypervolume history

In [ ]:
df = pareto.hypervolume_history(df)
pareto.plot_hypervolume_history(df)

In [ ]:
_ = pareto.plot_correlation_heatmap(df)

In [ ]:
_ = pareto.plot_objective_correlations(df)

## Observed three-objective Pareto front

Restricted to simulations with physically valid `Y_TDT`, `Y_rhoRDT`, and `Y_minus_peak_IFAR`.

In [ ]:
valid_mask = (
    (df["Y_TDT"] > 0)
    & (df["Y_rhoRDT"] > 0)
    & np.isfinite(df["Y_minus_peak_IFAR"])
)
valid_df, pareto_df = pareto.pareto_front(df, valid_mask=valid_mask)
pareto.plot_pareto_3d(valid_df, pareto_df)

### Inspect a Pareto (non-dominated) point

The table below gives each Pareto point both its plotted DataFrame index and its original database simulation index (`index_0`). Use either index with `pareto.inspect_dominating_point(...)` to recover the corresponding design inputs.

In [ ]:
pareto_table = pareto.list_dominating_points(pareto_df)
pareto_table.to_csv("../results/pareto_dominating_points.csv", index=False)
print("Saved as: ../results/pareto_dominating_points.csv")
pareto_table

In [ ]:
# Examples:
# result = pareto.inspect_dominating_point(pareto_df, 35, index_type="df_index")
# result = pareto.inspect_dominating_point(pareto_df, 35, index_type="simulation_index")
# result = pareto.inspect_dominating_point(pareto_df, 0, index_type="pareto_position")

## Pairwise 2D Pareto fronts (recomputed per objective pair)

In [ ]:
pareto.plot_pairwise_objective_fronts(valid_df)

## DT ice-layer thickness vs. drive current (all valid simulations)

In [ ]:
_ = pareto.plot_dt_ice_thickness_vs_current(df)